## Chapter 4: Training Models

### 1. Linear Regression & The Normal Equation
To understand Linear Regression, we first generate some linear-looking data with random noise based on the equation $y = 4 + 3x_1 + \text{noise}$. 

Our goal is to find the best parameters (weights), represented as $\theta_0$ (intercept) and $\theta_1$ (slope), that fit a straight line to this data. The mathematical shortcut to find these optimal weights directly is called the **Normal Equation**:
$$\hat{\boldsymbol{\theta}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import add_dummy_feature

# 1. Generate linear-looking data
rng = np.random.default_rng(seed=42)
m = 200 # number of instances
X = 2 * rng.random((m, 1)) # column vector
y = 4 + 3 * X + rng.standard_normal((m, 1)) # column vector

# 2. Compute theta using the Normal Equation
# We add x0 = 1 to each instance to account for the bias term (intercept)
X_b = add_dummy_feature(X) 
theta_best = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

print(theta_best)
# Expected Output: ~ [[3.69], [3.32]] 
# (Close to our original equation parameters 4 and 3)
```

### 2. Making Predictions & Visualization
Now that we have the optimal weights (`theta_best`), we can make predictions for new data points and plot the regression line.

```python
# Create two new points (x=0 and x=2) to draw the line
X_new = np.array([[0], [2]])
X_new_b = add_dummy_feature(X_new)

# Predict y values using the dot product
y_predict = X_new_b @ theta_best

# Plot the model's predictions as a red line
plt.figure(figsize=(6, 4))
plt.plot(X_new, y_predict, "r-", label="Predictions")
plt.plot(X, y, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([0, 2, 0, 15])
plt.grid()
plt.legend(loc="upper left")
plt.show()
```

### 3. Scikit-Learn & The Pseudoinverse Approach
Using `np.linalg.inv()` for the Normal Equation is mathematically sound, but in the real world, it can fail (e.g., if the matrix is singular/non-invertible). 

Scikit-Learn's `LinearRegression` class avoids this by using a more stable and efficient approach called **Singular Value Decomposition (SVD)**. It computes the Moore-Penrose pseudoinverse ($\mathbf{X}^+$) instead of the standard inverse.

```python
from sklearn.linear_model import LinearRegression

# Train the model using Scikit-Learn (SVD approach)
lin_reg = LinearRegression()
lin_reg.fit(X, y)

print(f"Intercept: {lin_reg.intercept_}")
print(f"Coefficient: {lin_reg.coef_}")

# Alternatively, you can compute the pseudoinverse directly using NumPy:
# theta_best_svd = np.linalg.pinv(X_b) @ y
```

## 4. Batch Gradient Descent

Unlike the Normal Equation which calculates the exact solution in one shot, **Gradient Descent** is an iterative optimization algorithm. It starts with random weights (random initialization) and tweaks them step-by-step to minimize the cost function (MSE). 

The term **Batch** means that the algorithm uses the *entire* training set ($\mathbf{X}$) to compute the gradients at every single step.

### The Algorithm Implementation
The mathematical formula for the gradient vector of the cost function is:
$$\nabla_{\boldsymbol{\theta}} \text{MSE}(\boldsymbol{\theta}) = \frac{2}{m} \mathbf{X}^T (\mathbf{X} \boldsymbol{\theta} - \mathbf{y})$$

Once we have the gradient, we update our weights by taking a step in the opposite direction. The size of this step is determined by the learning rate ($\eta$):
$$\boldsymbol{\theta}^{(\text{next step})} = \boldsymbol{\theta} - \eta \nabla_{\boldsymbol{\theta}} \text{MSE}(\boldsymbol{\theta})$$

```python
import numpy as np

# Set the hyperparameters
eta = 0.1 # learning rate
n_epochs = 1000 # number of iterations
m = len(X_b) # number of instances

# 1. Random Initialization
rng = np.random.default_rng(seed=42)
theta = rng.standard_normal((2, 1)) 

# 2. The Training Loop
for epoch in range(n_epochs):
    # Calculate the gradient over the entire dataset
    gradients = 2 / m * X_b.T @ (X_b @ theta - y)
    
    # Update the weights
    theta = theta - eta * gradients

print("Trained parameters (Batch Gradient Descent):")
print(theta)
# Expected Output: ~ [[3.69], [3.32]] (Same as the Normal Equation!)
```

### The Impact of the Learning Rate ($\eta$)
The learning rate is the most critical hyperparameter in Gradient Descent. If it's set incorrectly, the model might never learn. We can visualize this by plotting the first 20 steps of the algorithm for different learning rates.

```python
import matplotlib.pyplot as plt
import matplotlib as mpl

def plot_gradient_descent(theta, eta):
    m = len(X_b)
    plt.plot(X, y, "b.")
    n_epochs = 1000
    n_shown = 20
    theta_path = []
    
    for epoch in range(n_epochs):
        if epoch < n_shown:
            y_predict = X_new_b @ theta
            color = mpl.colors.rgb2hex(plt.cm.OrRd(epoch / n_shown + 0.15))
            plt.plot(X_new, y_predict, linestyle="solid", color=color)
            
        gradients = 2 / m * X_b.T @ (X_b @ theta - y)
        theta = theta - eta * gradients
        theta_path.append(theta)
        
    plt.xlabel("$x_1$")
    plt.axis([0, 2, 0, 15])
    plt.grid()
    plt.title(fr"$\eta = {eta}$")
    return theta_path

# Plotting the comparison
rng = np.random.default_rng(seed=42)
theta = rng.standard_normal((2, 1))

plt.figure(figsize=(10, 4))
plt.subplot(131); plot_gradient_descent(theta, eta=0.02); plt.ylabel("$y$", rotation=0)
plt.subplot(132); theta_path_bgd = plot_gradient_descent(theta, eta=0.1)
plt.subplot(133); plot_gradient_descent(theta, eta=0.5)
plt.show()
```

### Visual Analysis of Learning Rates
* **$\eta = 0.02$ (Too Low):** The model is learning, but taking tiny steps. It will eventually reach the solution, but it will take a very long time.
* **$\eta = 0.1$ (Optimal):** The model takes well-sized steps and converges perfectly to the optimal solution in just a few iterations.
* **$\eta = 0.5$ (Too High):** The algorithm steps are too large. It overshoots the minimum and completely diverges, jumping all over the place.

## 5. Stochastic Gradient Descent (SGD)

The main problem with Batch Gradient Descent is that it uses the whole training set to compute the gradients at every step, which makes it very slow when the training set is large. **Stochastic Gradient Descent (SGD)** solves this by picking a *random* single instance in the training set at every step and computing the gradients based only on that single instance.

This makes the algorithm much faster because it has very little data to manipulate at every iteration. However, due to its stochastic (random) nature, this algorithm is much less regular than Batch Gradient Descent. Instead of gently decreasing until it reaches the minimum, the cost function will bounce up and down, decreasing only on average. Over time it will end up very close to the minimum, but once it gets there it will continue to bounce around, never settling down.

### Implementing SGD with a Learning Schedule
To prevent the algorithm from bouncing around the minimum forever, we gradually reduce the learning rate. The steps start out large (which helps make quick progress and escape local minima), then get smaller and smaller, allowing the algorithm to settle at the global minimum. The function that determines the learning rate at each iteration is called the **learning schedule**.

```python
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

n_epochs = 50
t0, t1 = 5, 50  # learning schedule hyperparameters

def learning_schedule(t):
    return t0 / (t + t1)

rng = np.random.default_rng(seed=42)
theta = rng.standard_normal((2, 1))  # randomly initialized model parameters

n_shown = 20 # extra code
plt.figure(figsize=(6, 4)) # extra code

for epoch in range(n_epochs):
    for iteration in range(m):
        # extra code - generates figure
        if epoch == 0 and iteration < n_shown:
            y_predict = X_new_b @ theta
            color = mpl.colors.rgb2hex(plt.cm.OrRd(iteration / n_shown + 0.15))
            plt.plot(X_new, y_predict, color=color)
            
        random_index = rng.integers(m)
        xi = X_b[random_index : random_index + 1]
        yi = y[random_index : random_index + 1]
        
        # Calculate gradient using only one random instance
        gradients = 2 * xi.T @ (xi @ theta - yi) 
        
        # Update learning rate and theta
        eta = learning_schedule(epoch * m + iteration)
        theta = theta - eta * gradients
        
# extra code - beautifies figure
plt.plot(X, y, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([0, 2, 0, 15])
plt.grid()
plt.show()

print(theta)
# Expected Output: ~ [[3.69], [3.30]]
```

### SGD using Scikit-Learn
You don't have to implement SGD manually. Scikit-Learn provides the `SGDRegressor` class, which handles the learning schedule and stochastic nature for you automatically.

```python
from sklearn.linear_model import SGDRegressor

# Train the SGDRegressor
# max_iter: max number of epochs
# tol: stops training if the loss doesn't improve by at least this amount
# eta0: initial learning rate
sgd_reg = SGDRegressor(max_iter=1000, tol=1e-5, penalty=None, eta0=0.01,
                       n_iter_no_change=100, random_state=42)
                       
# fit() expects 1D targets, so we use y.ravel()
sgd_reg.fit(X, y.ravel())

print(sgd_reg.intercept_, sgd_reg.coef_)
# Expected Output: ~ [3.68] [3.33]
```

## 6. Mini-Batch Gradient Descent

**Mini-batch Gradient Descent** is the perfect middle ground between Batch and Stochastic Gradient Descent. Instead of computing the gradients based on the full training set (which is slow) or based on just one instance (which is erratic), Mini-batch computes the gradients on small, random sets of instances called **mini-batches**.

### The Algorithm Implementation
Here is how we implement it from scratch. Notice how we shuffle the dataset and slice it into batches of 20 instances.

```python
from math import ceil
import numpy as np

n_epochs = 50
minibatch_size = 20
n_batches_per_epoch = ceil(m / minibatch_size)

rng = np.random.default_rng(seed=42)
theta = rng.standard_normal((2, 1))

t0, t1 = 200, 1000 # Learning schedule hyperparameters

def learning_schedule(t):
    return t0 / (t + t1)

theta_path_mgd = []
for epoch in range(n_epochs):
    # Shuffle the dataset at the start of each epoch to ensure random batches
    shuffled_indices = rng.permutation(m)
    X_b_shuffled = X_b[shuffled_indices]
    y_shuffled = y[shuffled_indices]
    
    for iteration in range(0, n_batches_per_epoch):
        idx = iteration * minibatch_size
        
        # Pick the mini-batch (e.g., 20 instances)
        xi = X_b_shuffled[idx : idx + minibatch_size]
        yi = y_shuffled[idx : idx + minibatch_size]
        
        # Gradients are calculated ONLY on this specific mini-batch
        gradients = 2 / minibatch_size * xi.T @ (xi @ theta - yi)
        
        eta = learning_schedule(epoch * n_batches_per_epoch + iteration)
        theta = theta - eta * gradients
        theta_path_mgd.append(theta)
```

### Comparing the Three Algorithms
The main advantage of Mini-batch GD over Stochastic GD is that you can get a significant performance boost from hardware optimization of matrix operations, especially when using GPUs. 

Looking at the final plot in the parameter space ($\theta_0$ vs. $\theta_1$):
* **Batch (Blue):** Takes a perfectly smooth and direct path to the minimum, but computing each step is computationally expensive.
* **Stochastic (Red):** Bounces around erratically and wildly because every single instance pulls it in a different direction.
* **Mini-batch (Green):** Bounces around a bit, but walks a much straighter and more stable path than Stochastic, offering a fantastic balance of speed and stability.

## 7. Polynomial Regression

What if your data is actually more complex than a simple straight line? Surprisingly, you can still use a linear model to fit nonlinear data. A simple way to do this is to add powers of each feature as new features, then train a linear model on this extended set of features. This technique is called **Polynomial Regression**.

### Generating Nonlinear Data
First, let's generate some nonlinear data, based on a simple quadratic equation (plus some noise): 
$y = 0.5x^2 + x + 2 + \text{noise}$

```python
import numpy as np
import matplotlib.pyplot as plt

# Generate random quadratic data
rng = np.random.default_rng(seed=42)
m = 200 # number of instances
X = 6 * rng.random((m, 1)) - 3
y = 0.5 * X ** 2 + X + 2 + rng.standard_normal((m, 1))

# Plot the data
plt.figure(figsize=(6, 4))
plt.plot(X, y, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([-3, 3, 0, 10])
plt.grid()
plt.show()
```

### Fitting a Polynomial Model
A straight line will never fit this data properly. So, we use Scikit-Learn's `PolynomialFeatures` class to transform our training data, adding the square ($2^{\text{nd}}$ degree) of each feature in the training set as new features.

```python
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Transform the single feature X into two features: X and X^2
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly_features.fit_transform(X)

# Now, fit a LinearRegression model to this extended training data
lin_reg = LinearRegression()
lin_reg.fit(X_poly, y)

print(lin_reg.intercept_, lin_reg.coef_)
# Expected Output: ~ [2.00] [[1.11, 0.50]] 
# (Very close to our original equation: y = 0.5x^2 + 1.0x + 2.0)
```

### The Danger of High-Degree Polynomials (Overfitting)
If we perform high-degree Polynomial Regression, we will likely fit the training data much better than with plain Linear Regression. But is a better fit always a good thing? Let's compare a 300-degree polynomial model with a 2-degree model and a pure linear model.

```python
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

plt.figure(figsize=(6, 4))

# We test 3 different degrees: 1 (Linear), 2 (Quadratic), and 300 (Highly Polynomial)
for style, width, degree in (("r-+", 2, 1), ("b--", 2, 2), ("g-", 1, 300)):
    polybig_features = PolynomialFeatures(degree=degree, include_bias=False)
    std_scaler = StandardScaler()
    lin_reg = LinearRegression()
    
    # Build a pipeline to streamline the process
    polynomial_regression = make_pipeline(polybig_features, std_scaler, lin_reg)
    polynomial_regression.fit(X, y)
    
    # Generate new points to draw the continuous lines
    X_new = np.linspace(-3, 3, 100).reshape(100, 1)
    y_newbig = polynomial_regression.predict(X_new)
    
    label = f"{degree} degree{'s' if degree > 1 else ''}"
    plt.plot(X_new, y_newbig, style, label=label, linewidth=width)

plt.plot(X, y, "b.", linewidth=3)
plt.legend(loc="upper left")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([-3, 3, 0, 10])
plt.grid()
plt.show()
```

## 8. Learning Curves

If a model performs well on the training data but generalizes poorly according to the cross-validation metrics, it is overfitting. If it performs poorly on both, it is underfitting. Another way to tell is to look at the **Learning Curves**: these are plots of the model's performance on the training set and the validation set as a function of the training set size (or the training iteration).

### Diagnosing Underfitting
Let's look at the learning curves of a plain Linear Regression model (a simple straight line).

```python
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Compute learning curves for a simple Linear Regression model
train_sizes, train_scores, valid_scores = learning_curve(
    LinearRegression(), X, y, train_sizes=np.linspace(0.01, 1.0, 40), cv=5,
    scoring="neg_root_mean_squared_error")

# Convert negative RMSE to positive errors
train_errors = -train_scores.mean(axis=1)
valid_errors = -valid_scores.mean(axis=1)

# Plot the curves
plt.figure(figsize=(6, 4))
plt.plot(train_sizes, train_errors, "r-+", linewidth=2, label="train")
plt.plot(train_sizes, valid_errors, "b-", linewidth=3, label="valid")
plt.xlabel("Training set size")
plt.ylabel("RMSE")
plt.legend(loc="upper right")
plt.axis([0, 160, 0, 2.5])
plt.grid()
plt.show()
```
**Observation (Underfitting):** Both curves reach a plateau; they are close and typically high. If your model is underfitting the training data, adding more training examples will not help. You need to use a more complex model or come up with better features.

### Diagnosing Overfitting
Now let's look at the learning curves of a $10^{\text{th}}$-degree polynomial model on the exact same data.

```python
from sklearn.pipeline import make_pipeline

# Build a 10th-degree polynomial pipeline
polynomial_regression = make_pipeline(
    PolynomialFeatures(degree=10, include_bias=False),
    LinearRegression())

# Compute learning curves for the complex model
train_sizes, train_scores, valid_scores = learning_curve(
    polynomial_regression, X, y, train_sizes=np.linspace(0.01, 1.0, 40), cv=5,
    scoring="neg_root_mean_squared_error")

train_errors = -train_scores.mean(axis=1)
valid_errors = -valid_scores.mean(axis=1)

# Plot the curves
plt.figure(figsize=(6, 4))
plt.plot(train_sizes, train_errors, "r-+", linewidth=2, label="train")
plt.plot(train_sizes, valid_errors, "b-", linewidth=3, label="valid")
plt.xlabel("Training set size")
plt.ylabel("RMSE")
plt.legend(loc="upper right")
plt.axis([0, 160, 0, 2.5])
plt.grid()
plt.show()
```
**Observation (Overfitting):** 
1. The error on the training data is much lower than with the Linear Regression model.
2. There is a visible **gap** between the curves. This means that the model performs significantly better on the training data than on the validation data, which is the hallmark of an overfitting model. However, if you used a much larger training set, the two curves would continue to get closer.

## 9. Regularized Linear Models

As we saw with Polynomial Regression, a good way to reduce overfitting is to regularize the model (i.e., to constrain it). For a linear model, regularization is typically achieved by constraining the weights of the model. We will look at Ridge Regression, which implements this exact idea.

### Ridge Regression (L2 Regularization)
Ridge Regression is a regularized version of Linear Regression. A regularization term equal to $\frac{\alpha}{2} \sum_{i=1}^{n} \theta_i^2$ is added to the cost function during training. 

This forces the learning algorithm to not only fit the data but also keep the model weights as small as possible. The hyperparameter $\alpha$ controls how much you want to regularize the model. If $\alpha = 0$, then Ridge Regression is just plain Linear Regression. If $\alpha$ is very large, then all weights end up very close to zero and the result is a flat line going through the data's mean.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

# Generate a very small and noisy linear dataset
rng = np.random.default_rng(seed=42)
m = 20
X = 3 * rng.random((m, 1))
y = 1 + 0.5 * X + rng.standard_normal((m, 1)) / 1.5
X_new = np.linspace(0, 3, 100).reshape(100, 1)

# 1. Closed-form solution (Cholesky matrix factorization)
ridge_reg = Ridge(alpha=0.1, solver="cholesky")
ridge_reg.fit(X, y)
print("Ridge Prediction:", ridge_reg.predict([[1.5]]))
# Expected Output: ~ [[1.55]]
```

### Implementing Ridge using Stochastic Gradient Descent
Just as we used the Normal Equation and SGD for standard Linear Regression, we can apply Ridge regularization to SGD by adding the `penalty="l2"` hyperparameter. This simply means adding the squared value of the weights to the cost function.

```python
from sklearn.linear_model import SGDRegressor

# Train an SGD model with L2 Regularization (Ridge)
sgd_reg = SGDRegressor(penalty="l2", alpha=0.1 / m, tol=None,
                       max_iter=1000, eta0=0.01, random_state=42)

sgd_reg.fit(X, y.ravel()) # y.ravel() to flatten the target array
print("SGD L2 Prediction:", sgd_reg.predict([[1.5]]))
```

### The Math Behind Ridge Regression
Behind the scenes, the closed-form Normal Equation can be modified to include the $\alpha$ penalty term directly. Here is the mathematical formula computed via NumPy:
$\hat{\boldsymbol{\theta}} = (\mathbf{X}^T \mathbf{X} + \alpha \mathbf{A})^{-1} \mathbf{X}^T \mathbf{y}$

```python
# Demonstrating the underlying math of Ridge Regression
alpha = 0.1
A = np.array([[0., 0.], [0., 1.]]) # Identity matrix except for the bias term (index 0)
X_b = np.c_[np.ones((m, 1)), X]

# Compute theta using the modified Normal Equation
theta_ridge = np.linalg.inv(X_b.T @ X_b + alpha * A) @ X_b.T @ y
print("Manual Ridge Theta:\n", theta_ridge)
```

## 10. Bridging the Concepts: Regularization Deep Dive

If you have studied the theoretical foundations of Machine Learning (like Andrew Ng's courses), you already know how regularization works. Scikit-Learn simply uses different terminology for the exact same mathematical concepts.

### 1. The Terminology Translation
* **The Algorithm:** What theory calls standard *Regularization* is implemented in Scikit-Learn as **Ridge Regression** (also known as L2 Regularization).
* **The Hyperparameter:** Theoretical formulas use **$\lambda$ (Lambda)** to control the regularization strength. Scikit-Learn uses **$\alpha$ (Alpha)** for the exact same purpose.
* **The Parameters:** Theoretical weights denoted as **$w_j$** are equivalent to **$\theta_j$** in this book.

### 2. Ridge Regression (L2 Penalty)
Ridge Regression adds a penalty term equal to the **square** of the weights: $\sum \theta^2$. 
* **The Goal:** It forces the learning algorithm to shrink all parameters $\theta$ continuously.
* **The Effect:** On every step of Gradient Descent, the weights are multiplied by a number slightly less than $1$. They get very close to zero, smoothing out the curve and preventing overfitting, but they rarely ever become exactly zero.

### 3. Lasso Regression (L1 Penalty)
**Lasso** stands for *Least Absolute Shrinkage and Selection Operator*. Instead of penalizing the square of the weights, it adds a penalty equal to the **absolute value** of the weights: $\sum |\theta|$.
* **The Magic Feature:** Because of how absolute values behave mathematically, Lasso regularization completely eliminates the weights of the least important features (it sets them to **exactly $0$**). 
* **The Effect:** It acts as an automatic feature selector, outputting a sparse model where only the most important features remain active.

## 11. Lasso Regression (L1 Regularization)

**Lasso Regression** (Least Absolute Shrinkage and Selection Operator) is another regularized version of Linear Regression. Just like Ridge regression, it adds a regularization term to the cost function, but it uses the $\ell_1$ norm of the weight vector instead of half the square of the $\ell_2$ norm. 

An important characteristic of Lasso Regression is that it tends to completely eliminate the weights of the least important features (i.e., set them to zero). In other words, Lasso Regression automatically performs **feature selection** and outputs a sparse model.

### Scikit-Learn Implementation
Implementing Lasso is just as simple as implementing Ridge. You can use the `Lasso` class directly:

```python
from sklearn.linear_model import Lasso

# Train a Lasso model with alpha=0.1
lasso_reg = Lasso(alpha=0.1)
lasso_reg.fit(X, y)

print("Lasso Prediction:", lasso_reg.predict([[1.5]]))
# Expected Output: ~ [1.87]
```

### The Geometry of Regularization (Why Lasso sets weights to zero)
The complex matplotlib code in this section generates a beautiful visual comparison between L1 (Lasso) and L2 (Ridge) penalties in the parameter space (Figure 4-19):

* **L1 Penalty Shape (Lasso):** The penalty looks like a **diamond**. When Gradient Descent optimizes the cost function, the path tends to bounce down into the corners of this diamond. Because the corners lie exactly on the axes, one or more weights (like $\theta_2$) become exactly zero. The algorithm then slides down the axis to the global minimum.
* **L2 Penalty Shape (Ridge):** The penalty looks like a **circle**. The optimization path smoothly rolls down toward the center without getting trapped on the axes. Thus, the weights shrink and get very small, but rarely hit exactly zero.

*Note: When using Lasso, the Gradient Descent path can bounce around the optimum at the end. To ensure it converges perfectly, you should gradually reduce the learning rate during training (using a learning schedule).*

## 12. Elastic Net

**Elastic Net** is a middle ground between Ridge Regression and Lasso Regression. The regularization term is simply a mix of both Ridge and Lasso's regularization terms, and you can control the mix ratio $r$ (in Scikit-Learn, this is called `l1_ratio`). 
* When $r = 0$, Elastic Net is equivalent to Ridge Regression.
* When $r = 1$, it is equivalent to Lasso Regression.

### Which Regression Algorithm Should You Use?
Here is a practical checklist based on standard Machine Learning rules of thumb:
1. **Plain Linear Regression:** Almost never. It is generally preferable to have at least a little bit of regularization.
2. **Ridge Regression:** This is your solid **default**. Use this if you are unsure.
3. **Lasso Regression:** Use this if you suspect that only a few features are actually useful. It will automatically reduce useless features' weights down to exactly zero.
4. **Elastic Net:** Generally preferred over Lasso. Lasso can behave erratically when the number of features is greater than the number of training instances, or when several features are strongly correlated. Elastic Net brings the feature selection of Lasso with the stability of Ridge.

### Scikit-Learn Implementation
Implementing Elastic Net is straightforward. We set the `l1_ratio` to 0.5 to get a 50/50 mix of Ridge and Lasso penalties.

```python
from sklearn.linear_model import ElasticNet

# Train an Elastic Net model
# l1_ratio=0.5 means it behaves half like Lasso and half like Ridge
elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic_net.fit(X, y)

print("Elastic Net Prediction:", elastic_net.predict([[1.5]]))
# Expected Output: ~ [1.86]
```

## 13. Early Stopping

A completely different way to regularize iterative learning algorithms (such as Gradient Descent) is to simply stop training as soon as the validation error reaches a minimum. This is called **Early Stopping**. Geoffrey Hinton called it a "beautiful free lunch" because it is highly effective, simple to understand, and computationally cheap.

### The Intuition
As the epochs go by, the algorithm learns, and its prediction error (RMSE) on the training set goes down, along with its prediction error on the validation set. 
However, after a while, the validation error stops decreasing and starts to go back up. This indicates that the model has started to overfit the training data. With Early Stopping, you just stop training as soon as the validation error reaches the absolute minimum.

### Implementation with Scikit-Learn
To implement this, we can use `SGDRegressor` with `partial_fit()`, which allows us to train the model one step at a time inside a loop. We evaluate the model at each step and save a copy of the best model we've seen so far.

```python
from copy import deepcopy
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDRegressor
import numpy as np
import matplotlib.pyplot as plt

# 1. Prepare a highly complex polynomial pipeline to force overfitting
preprocessing = make_pipeline(PolynomialFeatures(degree=90, include_bias=False),
                              StandardScaler())
X_train_prep = preprocessing.fit_transform(X_train)
X_valid_prep = preprocessing.transform(X_valid)

# 2. Initialize the SGDRegressor
# penalty=None because we are using Early Stopping for regularization, not L2/L1
sgd_reg = SGDRegressor(penalty=None, eta0=0.002, random_state=42)

n_epochs = 500
best_valid_rmse = float('inf')
best_model = None

# 3. The Early Stopping Loop
for epoch in range(n_epochs):
    # partial_fit continues training where it left off, rather than starting from scratch
    sgd_reg.partial_fit(X_train_prep, y_train)
    
    y_valid_predict = sgd_reg.predict(X_valid_prep)
    val_error = root_mean_squared_error(y_valid, y_valid_predict)
    
    # Save the model if it beats our best record
    if val_error < best_valid_rmse:
        best_valid_rmse = val_error
        best_model = deepcopy(sgd_reg)

print("Best Validation RMSE achieved:", best_valid_rmse)
```

*Note: In practice, validation curves can be noisy, so they might bounce up and down a bit before genuinely rising. A robust implementation (like the one built into Scikit-Learn) often stops training only after the validation error has been strictly above the minimum for a certain number of epochs (controlled by the `n_iter_no_change` hyperparameter).*

## 14. Logistic Regression

Despite its confusing name, **Logistic Regression** is widely used for **classification** tasks, not regression. Just like a Linear Regression model, it computes a weighted sum of the input features (plus a bias term). However, instead of outputting the result directly, it outputs the *logistic* of this result.

### 1. Estimating Probabilities & The Sigmoid Function
The logistic—also called the logit, noted as $\sigma(\cdot)$—is a sigmoid function (an S-shaped function) that outputs a number strictly between $0$ and $1$. It acts as a "probability squasher".

$$\sigma(t) = \frac{1}{1 + e^{-t}}$$

```python
import numpy as np
import matplotlib.pyplot as plt

# Visualizing the Sigmoid Function
lim = 6
t = np.linspace(-lim, lim, 100)
sig = 1 / (1 + np.exp(-t))

plt.figure(figsize=(8, 3))
plt.plot([-lim, lim], [0, 0], "k-")
plt.plot([-lim, lim], [0.5, 0.5], "k:")
plt.plot([-lim, lim], [1, 1], "k:")
plt.plot([0, 0], [-1.1, 1.1], "k-")
plt.plot(t, sig, "b-", linewidth=2, label=r"$\sigma(t) = \frac{1}{1 + e^{-t}}$")
plt.xlabel("t")
plt.legend(loc="upper left")
plt.axis([-lim, lim, -0.1, 1.1])
plt.gca().set_yticks([0, 0.25, 0.5, 0.75, 1])
plt.grid()
plt.show()
```

### 2. Decision Boundaries (The Iris Dataset)
To illustrate Logistic Regression, we use the famous **Iris dataset**, which contains the sepal and petal length and width of 150 iris flowers of three different species: *Iris setosa*, *Iris versicolor*, and *Iris virginica*.

First, let's try to build a binary classifier to detect the *Iris virginica* type based strictly on one feature: the **petal width**.

```python
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Load the data
iris = load_iris(as_frame=True)
X = iris.data[["petal width (cm)"]].values
y = iris.target_names[iris.target] == 'virginica' # 1 if Virginica, else 0

# Split the data and train the model
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)

# Predict probabilities for flowers with petal widths varying from 0 to 3 cm
X_new = np.linspace(0, 3, 1000).reshape(-1, 1)
y_proba = log_reg.predict_proba(X_new)
decision_boundary = X_new[y_proba[:, 1] >= 0.5][0, 0]

print(f"Decision Boundary: {decision_boundary:.3f} cm")
# The model predicts Virginica if petal width > ~1.65 cm
```

### 3. Visualizing Multiple Features
When we train the Logistic Regression model on **two features** (e.g., petal length and petal width), the decision boundary is no longer a single point on a line, but rather a straight line separating the two classes in a 2D space. The model outputs varying probability contours indicating how confident it is about a flower belonging to the *Virginica* class based on its coordinates.

## 15. Softmax Regression (Multinomial Logistic Regression)

The Logistic Regression model can be generalized to support multiple classes directly, without having to train and combine multiple binary classifiers (like One-vs-All). This is called **Softmax Regression**, or Multinomial Logistic Regression.

### How it Works
Instead of using a Sigmoid function that outputs a single probability for a binary choice, the model computes a raw score for *each* class. Then, it passes these scores through the **Softmax function**. The Softmax function normalizes these scores, transforming them into probabilities that strictly sum up to 1 (100%). The model predicts the class with the highest estimated probability.

### Scikit-Learn Implementation
In Scikit-Learn, the `LogisticRegression` model automatically switches to Softmax Regression when you train it on a dataset with more than two classes. 

Let's use the Iris dataset again, but this time we will train the model on **all three classes** instead of just separating *Virginica* from the rest.

```python
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 1. Load data (Using all 3 classes: Setosa, Versicolor, Virginica)
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = iris["target"] # Target contains 0, 1, and 2
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# 2. Train the Softmax Regression model
# Note: 'C' is the inverse of regularization strength (like 1/alpha).
# Higher C = less regularization.
softmax_reg = LogisticRegression(C=30, random_state=42)
softmax_reg.fit(X_train, y_train)

# 3. Make a prediction for a flower with length 5 and width 2
print("Predicted Class:", softmax_reg.predict([[5, 2]]))
# Expected Output: [2] (which corresponds to Iris Virginica)

print("Class Probabilities:", softmax_reg.predict_proba([[5, 2]]).round(2))
# Expected Output: [[0.0  0.04  0.96]]
# Meaning: 0% Setosa, 4% Versicolor, 96% Virginica (Sums to 1.0)
```

### Visualizing Multiclass Decision Boundaries
Because there are three classes, the decision boundaries are no longer a single line. The model creates regions for each class. 

*(The code below generates Figure 4-25, showing the beautiful multi-class decision regions and the probability contours for the Virginica class).*

```python
# Generate data for contour plot
custom_cmap = ListedColormap(["#fafab0", "#9898ff", "#a0faa0"])
x0, x1 = np.meshgrid(np.linspace(0, 8, 500).reshape(-1, 1),
                     np.linspace(0, 3.5, 200).reshape(-1, 1))
X_new = np.c_[x0.ravel(), x1.ravel()]

y_proba = softmax_reg.predict_proba(X_new)
y_predict = softmax_reg.predict(X_new)

zz1 = y_proba[:, 1].reshape(x0.shape)
zz = y_predict.reshape(x0.shape)

plt.figure(figsize=(10, 4))
plt.plot(X[y == 2, 0], X[y == 2, 1], "g^", label="Iris virginica")
plt.plot(X[y == 1, 0], X[y == 1, 1], "bs", label="Iris versicolor")
plt.plot(X[y == 0, 0], X[y == 0, 1], "yo", label="Iris setosa")

plt.contourf(x0, x1, zz, cmap=custom_cmap)
contour = plt.contour(x0, x1, zz1, cmap="hot")
plt.clabel(contour, inline=1)
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="center left")
plt.axis([0.5, 7, 0, 3.5])
plt.grid()
plt.show()
```

## Chapter 4: Core Concepts & Exercise Solutions (Cheat Sheet)

These notes summarize the crucial theoretical takeaways from the Chapter 4 exercises.

### Part 1: Gradient Descent vs. Normal Equation
* **1. Millions of Features:** If you have millions of features, you **must** use Gradient Descent (Stochastic or Mini-batch). You **cannot** use the Normal Equation or SVD because their computational complexity grows quadratically, making them impossibly slow.
* **2. The Importance of Scaling:** If features have completely different scales, the GD cost function becomes an elongated bowl, making convergence painfully slow. **Always scale data for GD!** 
  * *Note:* The Normal Equation doesn't care about scaling. However, Regularized models (Ridge/Lasso) *do* require scaling, otherwise, large-scale features get penalized unfairly.

### Part 2: Gradient Descent Behaviors
* **3. Local Minima:** Gradient Descent **cannot** get stuck in a local minimum when training a Logistic or Linear Regression model because their cost functions are perfectly **convex** (bowl-shaped, meaning there is only one global minimum).
* **4. Do all GD algorithms converge to the exact same model?** No. While Batch GD converges exactly to the global minimum, Stochastic GD and Mini-batch GD bounce around it forever. To make them truly converge, you must gradually reduce the learning rate using a **learning schedule**.
* **7. Speed vs. Convergence:** 
  * *Fastest to the vicinity:* Stochastic GD (since it updates parameters using only 1 instance).
  * *Actually converges:* Only Batch GD.

### Part 3: Diagnosing Errors (Overfitting vs. Underfitting)
* **5. Rising Validation Error (Batch GD):** 
  * If the *training error is also rising*, your learning rate is too high (the model is diverging). Fix: Reduce the learning rate ($\eta$).
  * If the *training error is dropping*, your model is overfitting. Fix: Stop training (Early Stopping).
* **6. Early Stopping with SGD/Mini-batch:** Because these algorithms are random and bouncy, the validation error goes up and down locally. **Do not stop immediately** at the first rise. Instead, save the model at regular intervals and only stop when the error hasn't improved for a long time.
* **8. The "Gap" in Learning Curves:** A large gap between the training curve (low error) and validation curve (high error) means severe **Overfitting**. Three ways to fix it:
  1. Reduce model complexity (e.g., lower the polynomial degree).
  2. Add regularization (e.g., Ridge or Lasso).
  3. Feed the model more training data.
* **9. High and Equal Errors:** If both training and validation errors are high and close together, the model is **Underfitting (High Bias)**. To fix this in a regularized model, you must **reduce** the regularization hyperparameter ($\alpha$).

### Part 4: Choosing the Right Model
* **10. The Regularization Checklist:**
  * *Ridge vs. Plain Linear Regression:* Always prefer Ridge. Having at least a little regularization almost always improves generalization.
  * *Lasso vs. Ridge:* Use Lasso when you suspect only a few features are actually useful. Lasso acts as an automatic feature selector by pushing useless weights to exactly zero.
  * *Elastic Net vs. Lasso:* Lasso behaves erratically if features are highly correlated or outnumber the instances. Elastic Net gives you Lasso's feature selection without the erratic bugs (set `l1_ratio` close to 1).
* **11. Softmax vs. Multiple Logistic Regression:** 
  * If the classes are **mutually exclusive** (e.g., predicting if a picture is a Dog, Cat, *or* Bird), use ONE **Softmax** classifier.
  * If the classes are **not exclusive** (e.g., predicting if a picture is Outdoor/Indoor AND Daytime/Nighttime — since a picture can be both Outdoor and Daytime), you must train **TWO separate Logistic Regression** classifiers.

# Chapter 4: Training Models - The Big Picture & Workflow

This section provides a high-level summary of the theoretical concepts, optimization algorithms, and regularization techniques covered in Chapter 4. 

## 1. Linear Regression Optimization
There are two completely different ways to train a Linear Regression model:
*   **The Normal Equation (Closed-form):** A mathematical shortcut that computes the perfect weights in one step using matrix operations. 
    *   *Pros:* No hyperparameters (like learning rate) to tweak, exact answer.
    *   *Cons:* Very slow and computationally expensive if you have a large number of features (e.g., > 100,000).
*   **Gradient Descent (Iterative):** Tweaks the weights step-by-step to minimize the Cost Function (MSE). 
    *   *Crucial Rule:* You **must** scale your features (e.g., using `StandardScaler`) before using Gradient Descent, otherwise, it will take a very long time to converge.

## 2. The Gradient Descent Family
*   **Batch Gradient Descent:** Uses the *entire* dataset to calculate the gradient at each step. Smooth and exact, but terribly slow on large datasets.
*   **Stochastic Gradient Descent (SGD):** Picks just *one random instance* per step. Extremely fast and can jump out of local minima, but heavily bounces around and never truly settles unless you use a learning schedule.
*   **Mini-batch Gradient Descent:** The sweet spot. Evaluates a small, random batch of instances (e.g., 32) per step. It is much faster than Batch GD and more stable than SGD, and it benefits massively from hardware acceleration (GPUs).

## 3. Polynomial Regression & Learning Curves
When data is non-linear, we can add powers of features (e.g., $x^2, x^3$) using `PolynomialFeatures` to let a linear model fit nonlinear patterns.
*   **Diagnosing the Model:** We use **Learning Curves** (plotting training and validation errors over time) to check model health:
    *   **Underfitting (High Bias):** Both curves plateau at a high error. Adding more data won't help; you need a more complex model.
    *   **Overfitting (High Variance):** Training error is very low, but validation error is high (a visible gap between the curves). You need more data, a simpler model, or regularization.

## 4. Regularization (Fighting Overfitting)
Regularization penalizes large weights, forcing the model to be simpler and smoother.
*   **Ridge Regression ($L_2$ penalty):** The standard default. It shrinks all weights toward zero but rarely sets them to exactly zero.
*   **Lasso Regression ($L_1$ penalty):** Automatically performs feature selection by pushing the weights of useless features to **exactly zero**. Use this if you suspect only a few features actually matter.
*   **Elastic Net:** A mix of Ridge and Lasso. Safer and more stable than Lasso, especially when features are highly correlated.
*   **Early Stopping:** Stopping the Gradient Descent training loop the exact moment the validation error reaches its minimum, before it starts climbing back up.

## 5. Classification Algorithms
*   **Logistic Regression:** Used for binary classification. It passes the linear regression output through a **Sigmoid** function to squash the value between $0$ and $1$, outputting a probability.
*   **Softmax Regression:** Used for multiclass classification when classes are mutually exclusive (e.g., detecting if a flower is Setosa, Versicolor, OR Virginica). It computes a score for each class and normalizes them so they all sum up to $100\%$.